# Stage 4 — BM25 + Stage 3 dense hard-negative pilot

Approved validation-only seed-42 pilot. Searches the full 5,183-document corpus and selects exactly one highest-ranked passing BM25 negative plus one highest-ranked passing Stage-3-seed-42 dense negative per positive pair. It applies qrels exclusion, duplicate exclusion, and the unchanged 0.05 cosine margin, then runs a batch-32 smoke test before training. The Stage 4 student starts fresh from the pinned pretrained E5-small-v2 revision. Test files are never loaded. No three-seed run is performed.


In [ ]:
%pip install -q "sentence-transformers==6.0.1" "transformers==5.16.1" "faiss-cpu==1.15.0" "rank-bm25==0.2.2" datasets pandas


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import base64
import hashlib
import io
import zipfile
from pathlib import Path

PAYLOAD_SHA256 = 'FBD29698D8567AA6EF75A49EA1D040ED3D52F940DC1B5F521D34A01936341595'
PAYLOAD_B64 = '...'
payload = base64.b64decode(PAYLOAD_B64)
assert hashlib.sha256(payload).hexdigest().upper() == PAYLOAD_SHA256
SOURCE_DIR = Path('/content/drive/MyDrive/RAFT/stage4_hardneg/source_snapshot')
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(SOURCE_DIR)
print(f'STAGE4_SOURCE_READY={SOURCE_DIR}')
print(f'PAYLOAD_SHA256={PAYLOAD_SHA256}')


In [ ]:
import json
import subprocess
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/RAFT')
CONFIG = SOURCE_DIR / 'configs/stage4_hardneg_pilot.json'
DATA_DIR = DRIVE_ROOT / 'stage1_scifact/processed'
STAGE3_RUN = DRIVE_ROOT / 'stage3_inbatch/final_seed42'
required = [
    DATA_DIR / 'corpus.jsonl',
    DATA_DIR / 'queries_train.jsonl',
    DATA_DIR / 'qrels_train.tsv',
    DATA_DIR / 'queries_validation.jsonl',
    DATA_DIR / 'qrels_validation.tsv',
    DRIVE_ROOT / 'stage2_baselines/results/stage2_baselines.csv',
    DRIVE_ROOT / 'stage3_inbatch/final_results/stage3_final_summary.json',
    STAGE3_RUN / 'results/stage3_final_result.json',
    STAGE3_RUN / 'best_checkpoint',
    STAGE3_RUN / 'embeddings/corpus_embeddings.npy',
    STAGE3_RUN / 'embeddings/corpus_ids.json',
    STAGE3_RUN / 'embeddings/cache_metadata.json',
    STAGE3_RUN / 'embeddings/stage3_inbatch_seed42.index.faiss',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Missing required Drive artifacts: {missing}'
assert torch.cuda.is_available(), 'Select a Colab GPU runtime before running Stage 4.'
config = json.loads(CONFIG.read_text())
assert config['training']['batch_size'] == 32
assert config['training']['effective_batch_size'] == 32
assert config['training']['gradient_accumulation_steps'] == 1
assert config['training']['learning_rate'] == 5e-6
assert config['mining']['candidate_pool_size_per_method'] == 5183
assert config['mining']['candidate_search_scope'] == 'full_corpus'
assert config['mining']['hard_negatives_by_source'] == {'bm25': 1, 'stage3_seed42_dense': 1}
assert config['mining']['false_negative_filter']['cosine_margin'] == 0.05
assert config['mining']['false_negative_filter']['relax_filter_if_insufficient'] is False
assert config['mining']['false_negative_filter']['stop_if_no_full_corpus_candidate_passes'] is True
assert config['evaluation']['load_or_read_test_split'] is False
print(f'GPU={torch.cuda.get_device_name(0)}')
print('STAGE4_FULL_CORPUS_PREFLIGHT_VALIDATION_ONLY_READY')


In [ ]:
subprocess.run([
    sys.executable,
    str(SOURCE_DIR / 'src/run_stage4_pilot.py'),
    '--repo-dir', str(SOURCE_DIR),
    '--drive-root', str(DRIVE_ROOT),
], check=True)


In [ ]:
import pandas as pd

MINING_SUMMARY = DRIVE_ROOT / 'stage4_hardneg/mined/stage4_seed42_mining_summary.json'
SMOKE_RESULT = DRIVE_ROOT / 'stage4_hardneg/smoke_seed42/results/stage4_smoke_result.json'
PILOT_RESULT = DRIVE_ROOT / 'stage4_hardneg/pilot_seed42/results/stage4_pilot_result.json'
COMPARISON = DRIVE_ROOT / 'stage4_hardneg/pilot_seed42/results/stage4_pilot_validation_comparison.csv'
mining = json.loads(MINING_SUMMARY.read_text())
smoke = json.loads(SMOKE_RESULT.read_text())
pilot = json.loads(PILOT_RESULT.read_text())
assert mining['data']['validation_loaded'] is False
assert mining['data']['test_split_loaded'] is False
assert mining['selection']['candidate_search_scope'] == 'full_corpus'
assert mining['selection']['candidate_pool_size_per_method'] == 5183
assert mining['selection']['filter_relaxed'] is False
assert smoke['passed'] is True and smoke['global_steps_completed'] == 3
assert smoke['config']['training']['batch_size'] == 32
assert pilot['test_split_loaded'] is False
assert pilot['fresh_pretrained_initialization'] is True
print('SMOKE TEST: PASSED AT BATCH 32')
print('\nMINING SUMMARY:')
print(json.dumps(mining['selection'], indent=2, sort_keys=True))
print('\nVALIDATION COMPARISON:')
display(pd.read_csv(COMPARISON))
print('\nLOSS SUMMARY:')
print(json.dumps(pilot['loss_summary'], indent=2, sort_keys=True))
print('\nVALIDATION EVALUATIONS:')
print(json.dumps(pilot['validation_evaluations'], indent=2, sort_keys=True))
print(f"\nBEST CHECKPOINT: {pilot['saved_best_checkpoint']}")
print(f"CORPUS CACHE: {DRIVE_ROOT / 'stage4_hardneg/pilot_seed42/embeddings'}")
print(f"TEST SPLIT LOADED: {pilot['test_split_loaded']}")
print(f'Attach or paste this result in the Codex task: {PILOT_RESULT}')
